# Paradigmi di apprendimento alternativi
(3 esercizi)

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision import models, datasets, transforms
import torch.nn.functional as F

## 1: Implementazione Transfer learning

Obiettivo: Fare transfer learning con una rete ResNet18 pre-addestrata su ImageNet e riaddestrarla su CIFAR10

- Usare ResNet18 pre-addestrata su ImageNet
- Crea un layer fully connected per sostituire l’ultimo della ResNet18 per far sì che le reti
abbiamo lo stesso numero di classi di CIFAR10

In [2]:
def train_model(model, train_loader, optimizer, criterion, device, epochs=5):
  model.train()
  for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in train_loader:
      images, labels = images.to(device), labels.to(device)

      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()

  print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

def evaluate_model(model, test_loader, device):
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    for images, labels in test_loader:
      images, labels = images.to(device), labels.to(device)

      outputs = model(images)
      _, predicted = torch.max(outputs.data, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()
    print(f"Accuracy sul test set: {100 * correct / total:.2f}%")

In [3]:
# Trasformazioni per il preprocessing dei dati
transform = transforms.Compose([
    transforms.Resize(224),  # CIFAR10 ha immagini 32x32, ridimensioniamo a 224x224 per ResNet
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Caricamento del dataset CIFAR10
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

# Caricamento del modello ResNet18 pre-addestrato
model = models.resnet18(pretrained=True)
# model = timm.create_model("resnet18", pretrained=True, num_classes=10)

# Modifica del fully connected layer per adattarlo a CIFAR10 (10 classi)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

# Definizione del dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Congelamento dei pesi pre-addestrati (solo la fully connected layer sarà aggiornata)
for param in model.parameters():
    param.requires_grad = False

# Solo i parametri del fully connected layer saranno aggiornati
for param in model.fc.parameters():
    param.requires_grad = True

# Definizione loss function ottimizzatore e training/validation loop
criterion = nn.CrossEntropyLoss()
optmizer = optim.Adam(model.fc.parameters(), lr=0.001)

train_model(model, train_loader, optmizer, criterion, device, epochs=1)
evaluate_model(model, test_loader, device)


100%|██████████| 170M/170M [00:05<00:00, 33.8MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s]


Epoch [1/1], Loss: 0.8260
Accuracy sul test set: 78.57%


## 2: Implementazione Fine-Tuning

Obiettivo: Fare fine-tuning con la rete implementata nell’esercizio precedente

- Usare ResNet18 pre-addestrata su ImageNet
- Crea un layer fully connected per sostituire l’ultimo della ResNet18 per far sì che le reti abbiamo lo stesso numero di classi di CIFAR10

In [4]:
# Trasformazioni per il preprocessing dei dati
transform = transforms.Compose([
    transforms.Resize(224),  # CIFAR10 ha immagini 32x32, ridimensioniamo a 224x224 per ResNet
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Caricamento del dataset CIFAR10
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

# Caricamento del modello ResNet18 pre-addestrato
model = models.resnet18(pretrained=True)
# model = timm.create_model("resnet18", pretrained=True, num_classes=10)

# Modifica del fully connected layer per adattarlo a CIFAR10 (10 classi)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

# Definizione del dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Congelamento dei pesi pre-addestrati (solo la fully connected layer sarà aggiornata)
for param in model.parameters():
    param.requires_grad = True

# Definizione loss function ottimizzatore e training/validation loop
criterion = nn.CrossEntropyLoss()
optmizer = optim.Adam(model.fc.parameters(), lr=0.001)

train_model(model, train_loader, optmizer, criterion, device, epochs=1)
evaluate_model(model, test_loader, device)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch [1/1], Loss: 0.8247
Accuracy sul test set: 78.23%


## 3: Implementazione della Knowledge Distillation

Obiettivo: Distillare la conoscenza da una rete teacher pre-addestrata su ImageNet in una più piccola e leggera

- Usare ResNet50 pre-addestrata su ImageNet
- Crea un layer fully connected per sostituire l’ultimo della ResNet50 per far sì che le reti abbiamo lo stesso numero di uscite (andrebbe riaddestrato ma lasciamo stare)
- Creare una rete student fatta da zero
- Crea una funzione di distillazioneche contiene le loss necessarie (Cross entropy e Kullback-Leibler Divergence)

In [11]:
# Funzione di loss per la distillazione della conoscenza
def distillation_loss(student_outputs, teacher_outputs, labels, temperature, alpha):
  '''
  Nel distillation, le probabilità dell'insegnante e dello studente vengono ammorbidite (softened) utilizzando un parametro chiamato temperatura (T). Questo avviene dividendo i logit (le uscite grezze della rete) per la temperatura:
  Alta temperatura: Produce distribuzioni di probabilità più piatte, che danno maggiore enfasi anche alle classi che hanno probabilità più basse, aiutando lo studente a imparare dai dettagli delle predizioni dell'insegnante.
  Bassa temperatura (T ≈ 1): Si avvicina alla distribuzione softmax standard, dove i valori dominanti hanno probabilità molto più alte rispetto agli altri.
  L'uso di una temperatura maggiore di 1 serve ad ammorbidire le probabilità in modo che lo studente non si concentri solo sulle classi con le massime probabilità, ma consideri anche le altre classi (con probabilità più basse)
  che contengono informazioni preziose

  Quando aumenti la temperatura, ammorbidisci le probabilità, ma questo riduce anche l'entità del gradiente che guida l'addestramento del modello studente.
  Per compensare questo effetto di riduzione, si moltiplica la loss di distillazione (calcolata sulla base delle probabilità ammorbidite) per temperature^2
  '''
  loss_ce = F.cross_entropy(student_outputs, labels)  # Cross-entropy standard
  loss_kl = nn.KLDivLoss()(F.softmax(student_outputs / temperature, dim=1),
                             F.softmax(teacher_outputs / temperature, dim=1)) * (temperature * temperature)
  return loss_ce * (1. - alpha) + loss_kl * alpha


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Modello Student ---
class StudentNet(nn.Module):
  def __init__(self):
    super(StudentNet, self).__init__()
    self.features = nn.Sequential(
      nn.Conv2d(3, 16, kernel_size=3, padding=1),
      nn.BatchNorm2d(16),
      nn.ReLU(),
      nn.MaxPool2d(2, 2),
      nn.Conv2d(16, 32, kernel_size=3, padding=1),
      nn.BatchNorm2d(32),
      nn.ReLU(),
      nn.MaxPool2d(2, 2)
    )
    self.classifier = nn.Sequential(
      nn.Linear(32*56*56, 128),
      nn.ReLU(),
      nn.Linear(128, 10)
    )

  def forward(self, x):
    x = self.features(x)
    x = x.view(x.size(0), -1)
    x = self.classifier(x)
    return x

# --- Modello Teacher (RenNet18 pre addestrata) ---
teacher_model = models.resnet18(pretrained=True)
teacher_model.fc = nn.Linear(512, 10)
teacher_model = teacher_model.to(device)

# Student
student_model = StudentNet().to(device)

# --- CIFAR-10 ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)

# Ottimizzatori e parametri
t_optimizer = optim.Adam(teacher_model.parameters(), lr=1e-4)
s_optimizer = optim.Adam(student_model.parameters(), lr=1e-3)
t_criterion = nn.CrossEntropyLoss()

temperature = 2.0
alpha = 0.5

# --- Allenamento Teacher (1 epoca) ---
def train_model(model, dataloader, optimizer, criterion, device, epochs=1):
  model.train()
  for epoch in range(epochs):
    running_loss = 0.0
    for data, target in dataloader:
      data, target = data.to(device), target.to(device)
      optimizer.zero_grad()
      outputs = model(data)
      loss = criterion(outputs, target)
      loss.backward()
      optimizer.step()
      running_loss += loss.item()
    print(f"Teacher Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(dataloader):.4f}")

train_model(teacher_model, train_loader, t_optimizer, t_criterion, device, epochs=1)

# --- Knowledge Distillation ---
teacher_model.eval()

for epoch in range(2):
  student_model.train()
  running_loss = 0.0

  for data, target in train_loader:
    data, target = data.to(device), target.to(device)
    s_optimizer.zero_grad()

    # Otteniamo le predizioni del teacher e dello student
    with torch.no_grad():
      t_outputs = teacher_model(data)

    s_output = student_model(data)
    loss = distillation_loss(s_output, t_outputs, target, temperature, alpha)
    loss.backward()
    s_optimizer.step()
    running_loss += loss.item()

print(f'Epoch [{epoch+1}/2], Distillation loss: {running_loss / len(train_loader):.4f}')

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Teacher Epoch [1/1] - Loss: 0.3232


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:3359: UserWarning: reduction: 'mean' divides the total loss by both the batch size and the support size.'batchmean' divides only by the batch size, and aligns with the KL div math definition.'mean' will be changed to behave the same as 'batchmean' in the next major release.
  warnings.warn(


Epoch [2/2], Distillation loss: 0.4872
